In [26]:
import pandas as pd
import numpy as np
import os

# Load price data
csv_path = os.path.join('..', 'data', 'raw', 'cost', 'pesticide_price.csv')
df = pd.read_csv(csv_path)

# Load spray records to get product types
spray_path = os.path.join('..', 'data', 'raw', 'cost', 'powdery_mildew_fungicide_record_database.csv')
spray_df = pd.read_csv(spray_path, encoding='utf-8')

# Fix data entry errors
spray_df['Product'] = spray_df['Product'].str.replace('Gramoxone SL 2 ', 'Gramoxone SL 2')
spray_df['Product'] = spray_df['Product'].str.replace('Class Act ', 'Class Act')
spray_df['Product'] = spray_df['Product'].str.replace('InterLock', 'Interlock')
spray_df = spray_df[spray_df['Product'] != '.']
spray_df = spray_df.dropna(subset=['Product'])

# Build product -> type mapping (most frequent type for each product)
type_map = spray_df.groupby('Product')['Type'].agg(lambda x: x.value_counts().index[0]).to_dict()

# Assign type to each product in price data
df['Type'] = df['Product'].map(type_map)

# Select the R Price columns
r_price_cols = [c for c in df.columns if c.startswith('R Price')]

# Compute mean and std dev across years for each product
df['Mean R Price'] = df[r_price_cols].mean(axis=1)
df['Std R Price'] = df[r_price_cols].std(axis=1)

# Build LaTeX table rows grouped by type with averages in section headers
section_order = ['Fungicide', 'Herbicide', 'Adjuvant']
grouped_rows = []
for section in section_order:
    section_df = df[df['Type'] == section].sort_values('Product')
    section_mean = section_df['Mean R Price'].mean()
    section_std = section_df['Mean R Price'].std()
    grouped_rows.append(rf"\hline")
    grouped_rows.append(rf"\textbf{{\textit{{{section}s}}}} & & \textbf{{{section_mean:.4f}}} & \textbf{{{section_std:.4f}}} \\")
    grouped_rows.append(rf"\hline")
    for _, row in section_df.iterrows():
        product = row['Product'].replace('&', r'\&').replace('_', r'\_')
        unit = row['Unit']
        mean_val = row['Mean R Price']
        std_val = row['Std R Price']
        std_str = f"{std_val:.4f}" if not np.isnan(std_val) else "--"
        grouped_rows.append(f"{product} & {unit} & {mean_val:.4f} & {std_str} \\\\")

latex_rows = "\n".join(grouped_rows)

latex_table = rf"""\begin{{table}}[ht]
\centering
\caption{{Mean real prices and standard deviations\textsuperscript{{b}} of pesticides and adjuvants used in hop powdery mildew management programs (2014--2022), expressed in January 2022 U.S. dollars per unit. Nominal prices were obtained from regional vendor quotes and deflated using the Bureau of Labor Statistics Producer Price Index for farm products.\textsuperscript{{a}}}}
\label{{tab:S2_pesticides}}
\footnotesize
\setlength{{\tabcolsep}}{{4pt}}
\begin{{tabular}}{{lccc}}
\hline
\textbf{{Pesticide/Adjuvant}} & \textbf{{Unit}} & \textbf{{Mean Real Price (\$)}} & \textbf{{Std. Dev.}} \\
{latex_rows}
\hline
\end{{tabular}}
\vspace{{4pt}}
\par
\raggedright
\scriptsize
\textsuperscript{{a}} Product names are registered trademarks of their respective manufacturers. Use of trade names does not imply endorsement by the authors or their affiliated institutions.\\
\textsuperscript{{b}} Standard deviations reported as -- indicate that price data were available for only a single year.
\end{{table}}"""

# Save to file
output_path = os.path.join('..', 'reports', 'table_S2_pesticides.tex')
with open(output_path, 'w') as f:
    f.write(latex_table)

print(f"LaTeX table saved to {output_path}")
print()
print(latex_table)

LaTeX table saved to ..\reports\table_S2_pesticides.tex

\begin{table}[ht]
\centering
\caption{Mean real prices and standard deviations\textsuperscript{b} of pesticides and adjuvants used in hop powdery mildew management programs (2014--2022), expressed in January 2022 U.S. dollars per unit. Nominal prices were obtained from regional vendor quotes and deflated using the Bureau of Labor Statistics Producer Price Index for farm products.\textsuperscript{a}}
\label{tab:S2_pesticides}
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lccc}
\hline
\textbf{Pesticide/Adjuvant} & \textbf{Unit} & \textbf{Mean Real Price (\$)} & \textbf{Std. Dev.} \\
\hline
\textbf{\textit{Fungicides}} & & \textbf{2.5664} & \textbf{3.4034} \\
\hline
Accrue & oz & 1.3946 & -- \\
Copper-Count-N & fl oz & 0.2279 & 0.0346 \\
Flint & oz & 10.8797 & 2.2130 \\
Folpan & oz & 0.2936 & -- \\
Kaligreen & oz & 0.7693 & 0.1121 \\
Kocide 2000 & oz & 0.6762 & 0.1687 \\
Luna Sensation & oz & 8.9865 & 1.5964 \\
Milstop &

In [14]:
r_price_cols

['R Price 2014',
 'R Price 2015',
 'R Price 2016',
 'R Price 2017',
 'R Price 2018',
 'R Price 2019',
 'R Price 2020',
 'R Price 2021',
 'R Price 2022']